# Notebook 4 : Station spatiale internationale 🛰️

In [1]:
# Décommenter la ligne suivante pour installer les dépendances
#%pip install folium jupyter_bokeh nbconvert panel watchfiles

In [2]:
import folium
import pandas as pd
import panel as pn
import requests
import ast
from bs4 import BeautifulSoup

pn.extension()

La [station spatiale internationale](https://fr.wikipedia.org/wiki/Station_spatiale_internationale) (*ISS*) est en orbite autour de notre planète à une altitude légérement supérieure à 400 km. Cette station effectue une quinzaine de révolutions par jour et nous pouvons suivre [sa position en temps réel](https://spotthestation.nasa.gov/tracking_map.cfm) grâce à différents outils dont l'API de [Where the ISS at](https://wheretheiss.at/). Nous proposons dans la suite de mettre en place une application web de visualisation des données de l'ISS.

## Préliminaires

1. Utiliser la fonction `get` de `requests` pour récupérer les informations sur la position courante de l'ISS à partir de l'API de [Where the ISS at](https://wheretheiss.at/) à l'adresse suivante :<br/>[https://api.wheretheiss.at/v1/satellites/25544](https://api.wheretheiss.at/v1/satellites/25544)<br/>Comprendre en particulier à quoi correspond la valeur `timestamp`.

In [3]:
url_iss = "https://api.wheretheiss.at/v1/satellites/25544"



2. Écrire une fonction `get_iss_position` sans argument qui retourne les informations sur la position courante de l'ISS sous la forme d'un dictionnaire Python ou `None` si la requête échoue. Expliquer pourquoi cette fonction ne doit pas être décorée avec `pn.cache`.

In [4]:
def get_iss_position():
    r_iss = requests.get(url_iss)
    soup_iss = BeautifulSoup(r_iss.text, "html.parser")
    iss_pos = ast.literal_eval(soup_iss.text)
    return (iss_pos['latitude'], iss_pos['longitude'])

## Folium

Le module [Folium](https://python-visualization.github.io/folium/latest/) permet de visualiser des données géostatistiques dans une application web à partir de la bibliothèque JavaScript Leaflet.

3. Afficher une carte vierge du monde avec la fonction `Map` de `folium`.

In [5]:
m=folium.Map()
m

4. Récupérer la position de l'ISS dans un objet `iss_0` avec `get_iss_position` et afficher un marqueur (voir [la documentation](https://python-visualization.github.io/folium/latest/getting_started.html#Adding-markers)) sur la mappemonde.

In [6]:
iss_0 = get_iss_position()
iss_0

(43.31461306524, 111.01704158702)

In [7]:
m = folium.Map(zoom_start=10)
folium.Marker(
    location=[iss_0[0], iss_0[1]],
    tooltip="Click me!",
    popup="ISS",
).add_to(m)
m

5. Récupérer à nouveau la position de l'ISS dans un objet `iss_1` avec `get_iss_position` et afficher une ligne entre la position précédente et la nouvelle (voir [la documentation](https://python-visualization.github.io/folium/latest/getting_started.html#Vectors-such-as-lines)) sur la mappemonde.

In [8]:
iss_1 = get_iss_position()

trail_coordinates = [
    iss_0,
    iss_1
]

folium.PolyLine(trail_coordinates).add_to(m)
m

6. Récupérer à nouveau la position de l'ISS dans un objet `iss_2` avec `get_iss_position` et stocker les trois positions dans un DataFrame Pandas `iss_positions`.

In [ ]:
iss_2 = get_iss_position()

In [ ]:

iss_positions = pd.DataFrame([iss_0, iss_1, iss_2])
iss_positions.columns = ['latitude', 'longitude']
iss_positions[]

pandas.core.frame.DataFrame

7. Écrire une fonction `get_iss_map` qui prend un DataFrame tel que `iss_positions` en argument et retourne une carte Folium affichant la trajectoire de l'ISS avec une ligne et sa dernière position avec un marqueur.

In [ ]:
def get_iss_map(df):
    m=folium.Map()
    trail_coordinates = [
    iss_0,
    iss_1,
    ]

folium.PolyLine(trail_coordinates).add_to(m)

## Vers l'ISS et au-delà

8. Créer un *pane* `iss_df` de type `DataFrame` (voir [la documentation](https://panel.holoviz.org/reference/panes/DataFrame.html)) initialement vide et destiné à contenir un DataFrame Pandas avec trois colonnes `timestamp`, `latitude` et `longitude`.

9. Créer un widget `iss_button` de type `Button` (voir [la documentation](https://panel.holoviz.org/reference/widgets/Button.html)) et le lier à une fonction `update_position` qui sera appelée lorsque le bouton est pressé pour :

- récupérer la position courante de l'ISS avec `get_iss_position`,
- ajouter une ligne correspondante dans le DataFrame du *pane* `iss_df`,
- appeler `get_iss_map` et retourner le résultat.

## Application web

Nous pouvons maintenant mettre en place une application web qui pourra être démarrée avec la commande suivante (l'option `--allow-websocket-origin` n'est nécessaire que dans Onyxia) :
```{bash}
panel serve --autoreload --show --allow-websocket-origin=$(echo $VSCODE_PROXY_URI | cut -d '/' -f 3) notebooks/04_iss.ipynb
```

Les questions suivantes ont pour objet d'enrichir l'application au fur et à mesure. Il ne faut donc pas recréer une nouvelle application pour chaque question mais faire évoluer le code étape par étape. Il peut être utile d'ajouter de nouvelles cellules de code si besoin.

10. Mettre en forme une application à l'aide du modèle `FastListTemplate` (voir [la documentation](https://panel.holoviz.org/reference/templates/FastListTemplate.html)) avec le bouton `iss_button` et le *pane* `iss_df` dans la barre latérale et un *pane* Folium (voir [la documentation](https://panel.holoviz.org/reference/panes/Folium.html)) contenant la carte obtenue grâce à la fonction liée `update_position` dans la zone principale.

11. Utiliser le paramètre `sizing_mode` du *pane* Folium pour adapter sa taille à la fenêtre.

12. *(Bonus)* Adapter l'application pour afficher la vitesse de l'ISS dans un indicateur de type `Number` (voir [la documentation](https://panel.holoviz.org/reference/indicators/Number.html)).